In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import numpy as np
import shap 

df = pd.read_csv("dados_aula_01_exemplo_01.csv")

df["tem_filhos"] = (df["children"] > 0).astype(int)
df.head()

X = df.drop("charges", axis=1)
y = df["charges"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# print("numero de linhas e colunas dos dados de treino:", X_treino.shape)
# print("numero de linhas e colunas dos dados de treino:", X_teste.shape)

escalonador = StandardScaler()
categorizador = OneHotEncoder(drop='first', handle_unknown="ignore")
imputador_numero = SimpleImputer(strategy="median")
imputador_categorico = SimpleImputer(strategy="most_frequent")

# etapas_numericas
etapas_numericas = Pipeline(
    [
        ("imputer", imputador_numero),
        ("scaler", escalonador)
    ]
)


#etapas categoricas
etapas_categoricas = Pipeline(
    [
        ("imputer", imputador_categorico),
        ("encoder", categorizador)
    ]
)


# preprocessador = ColumnTransformer(
#     [
#         ("num", etapas_numericas, variaveis_numericas),
#         ("cat", etapas_categoricas, variaveis_categoricas)
#     ]
# )

preprocessador = ColumnTransformer(
    transformers=[
        ('numericas', etapas_numericas, ['age', 'bmi', 'children']),
        ('categoricas', etapas_categoricas, ['sex', 'smoker', 'region', 'tem_filhos'])
    ]
)

x_treino_transformado = preprocessador.fit_transform(X_treino)
x_teste_transformado = preprocessador.transform(X_teste)

modelo = DecisionTreeRegressor(random_state=42)

param_grid = {
    "max_depth": [2,3,5,7], # profundidade maxima da arvore
    "min_samples_split": [2,5,10,15,20,25] # numero minimo de amostras para dividir um nó
}

grid_search = GridSearchCV(
    estimator= modelo, # modelo a ser otimizado
    param_grid= param_grid, # grade de hiperparamentros
    cv=5, # 5-fold cross validation
    scoring="neg_root_mean_squared_error" # metricas de comparacao RMSE
)

grid_search.fit(x_treino_transformado, y_treino)

melhor_modelo = grid_search.best_estimator_
y_pred_treino = melhor_modelo.predict(x_treino_transformado)

mse_treino = root_mean_squared_error(y_treino, y_pred_treino)

print("Melhores parametros eencontrados:", grid_search.best_params_)
print("Melhores modelo eencontrados:", grid_search.best_estimator_)



y_pred_teste = melhor_modelo.predict(x_teste_transformado)
mse_teste = root_mean_squared_error(y_teste, y_pred_teste)

print("Root menad squared error no treino ", np.round(np.sqrt(mse_teste)))


Melhores parametros eencontrados: {'max_depth': 3, 'min_samples_split': 2}
Melhores modelo eencontrados: DecisionTreeRegressor(max_depth=3, random_state=42)
Root menad squared error no treino  69.0
